In [ ]:
# Check last executions
past_time = "12h"
notebook_client_name = "usdf_efd"

# N days for historical comparison
ndays = 8

# M1M3 Bump Test Log Error Analysis and Measured Forces

## Overview

This notebook is designed to facilitate the analysis of M1M3 force actuator “Bump Test” logs and the visualization of individual actuator performance.
It combines:

- **Bump Test Log Analysis:** Queries and processes recent bump test logs from the EFD to identify failed force actuators and extract relevant error information.

- **Script Execution Details:** Retrieves and displays execution logs for the `check_actuators.py` script, showing the duration, start/end times, and final statuses of each run.

- **Per-Actuator Visualization:** Plots measured and commanded forces, following errors, and other key metrics for individual actuators, helping users diagnose and understand any anomalies or deviations from expected values.

This integrated approach allows for efficient troubleshooting and performance assessment of M1M3 actuators based on either real-time or historical data.

## Setup and Imports

In [ ]:
# Imports and basic setup
import re
from astropy.time import Time, TimeDelta
from lsst.summit.utils.efdUtils import makeEfdClient
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

from matplotlib.ticker import FormatStrFormatter

from collections import defaultdict
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

from lsst.ts.xml.tables.m1m3 import (
    FATable,
    force_actuator_from_id,
)
from lsst.ts.xml.enums.MTM1M3 import BumpTest

# %matplotlib inline
# %load_ext lab_black

import asyncio
import ipywidgets as widgets
from IPython.display import display, clear_output

import nest_asyncio

nest_asyncio.apply()

## Utility Functions

### Converting Past Time Strings

In [ ]:
def convert_to_hours(past_time):
    """
    Convert a string that can be either a number of hours (e.g. '6h')
    or a number of days (e.g. '3d') to total number of hours (int).
    """
    match = re.match(r"(\d+)([dh]?)", past_time)
    if match:
        value, unit = match.groups()
        value = int(value)
        if unit == "d":
            return value * 24
        else:
            return value
    else:
        raise ValueError(
            "Invalid time format. Please use a format "
            "like '6h' for hours or '3d' for days."
        )

### Query and Filter Script Queue Logs

In [ ]:
async def query_script_queue_logs(
    start_time_str: str, end_time_str: str, client_name="usdf_efd"
):
    """
    Queries the log messages related to the script queue from the EFD.

    Parameters
    ----------
    start_time_str : str
        Start time in ISO format, e.g. "2024-11-04T12:00:00".
    end_time_str : str
        End time in ISO format, e.g. "2024-11-04T13:00:00".
    client_name : str, optional
        Name of the EFD client. Defaults to "usdf_efd".

    Returns
    -------
    pd.DataFrame
        DataFrame with script queue logs in the given time window.
    """
    # Convert string times to astropy Time
    start = Time(start_time_str, format="isot", scale="utc")
    end = Time(end_time_str, format="isot", scale="utc")

    # Create the EFD client
    possible_clients = ["summit_efd", "usdf_efd"]
    if client_name not in possible_clients:
        print(f"Invalid client name. Possible clients: {possible_clients}")
        return None

    client = makeEfdClient(client_name)
    script_logs = await client.select_time_series(
        topic_name="lsst.sal.ScriptQueue.logevent_script",
        fields="*",
        start=start,
        end=end,
    )

    return script_logs


def filter_and_process_queue_logs(script_logs):
    """
    Filters and processes the script logs DataFrame for a specific script (maintel/m1m3/check_actuators.py).

    Parameters
    ----------
    script_logs : pd.DataFrame
        DataFrame containing the script logs.

    Returns
    -------
    pd.DataFrame
        Processed DataFrame containing only relevant logs for the check_actuators script.
    """
    processState_mapping = {
        0: "UNKNOWN",
        1: "LOADING",
        2: "CONFIGURED",
        3: "RUNNING",
        4: "DONE",
        5: "LOADFAILED",
        6: "CONFIGURE_FAILED",
        7: "TERMINATED",
    }

    scriptState_mapping = {
        0: "UNKNOWN",
        1: "UNCONFIGURED",
        2: "CONFIGURED",
        3: "RUNNING",
        4: "PAUSED",
        5: "ENDING",
        6: "STOPPING",
        7: "FAILING",
        8: "DONE",
        9: "STOPPED",
        10: "FAILED",
        11: "CONFIGURE_FAILED",
    }

    # Script path and salIndex we’re focusing on
    path = "maintel/m1m3/check_actuators.py"
    salIndex = 1

    df_script_logs = pd.DataFrame(script_logs)

    # Filter logs for the specific script path and salIndex
    df_filtered_logs = df_script_logs[
        (df_script_logs["path"] == path) & (df_script_logs["salIndex"] == salIndex)
    ]

    # Reindex by private_rcvStamp, convert to UTC
    df_filtered_logs = df_filtered_logs.set_index("private_rcvStamp")
    df_filtered_logs.index = pd.to_datetime(df_filtered_logs.index, unit="s")
    df_filtered_logs.index = df_filtered_logs.index - timedelta(seconds=37)

    # Convert all columns starting with 'timestamp' to datetime, also shift TAI->UTC
    timestamp_columns = [
        col for col in df_filtered_logs.columns if col.startswith("timestamp")
    ]
    for col in timestamp_columns:
        df_filtered_logs[col] = pd.to_datetime(df_filtered_logs[col], unit="s")
        df_filtered_logs[col] = df_filtered_logs[col] - timedelta(seconds=37)

    # Map numeric process/script states to strings
    df_filtered_logs["processState_str"] = df_filtered_logs["processState"].map(
        processState_mapping
    )
    df_filtered_logs["scriptState_str"] = df_filtered_logs["scriptState"].map(
        scriptState_mapping
    )

    # Remove unneeded columns
    columns_to_remove = [
        "blockId",
        "blockSize",
        "cmdId",
        "executionId",
        "private_efdStamp",
        "private_kafkaStamp",
        "private_origin",
        "private_revCode",
        "private_sndStamp",
        "private_seqNum",
        "scriptBlockIndex",
        "isStandard",
        "private_identity",
        "processState",
        "scriptState",
    ]
    df_filtered_logs.drop(columns=columns_to_remove, inplace=True, errors="ignore")

    # Sort from most recent to oldest
    df_filtered_logs.sort_index(ascending=False, inplace=True)

    return df_filtered_logs


def extract_execution_details(df_check_actuators_log):
    """
    Extracts execution details from the script logs DataFrame.

    Parameters
    ----------
    df_check_actuators_log : pd.DataFrame
        DataFrame containing the filtered script logs.

    Returns
    -------
    pd.DataFrame
        DataFrame containing execution times, durations, and final statuses.
    """
    # Initialize an empty list to store execution details
    execution_data = []

    # Loop over each unique scriptSalIndex
    for script_sal_index in df_check_actuators_log["scriptSalIndex"].unique():
        df_sal = df_check_actuators_log[
            df_check_actuators_log["scriptSalIndex"] == script_sal_index
        ]
        df_sal = df_sal.sort_index()  # Ensure chronological order

        # Calculate start and end times for each execution
        start_time = df_sal["timestampProcessStart"].min()
        end_time = df_sal["timestampProcessEnd"].max()

        # Get the final process and script status
        final_process_status = df_sal["processState_str"].iloc[-1]
        final_script_status = df_sal["scriptState_str"].iloc[-1]

        execution_data.append(
            {
                "scriptSalIndex": script_sal_index,
                "start_time": start_time,
                "end_time": end_time,
                "FinalProcessStatus": final_process_status,
                "FinalScriptStatus": final_script_status,
            }
        )

    # Create the DataFrame
    df_executions = pd.DataFrame(execution_data)

    if not df_executions.empty:
        # Calculate duration in minutes
        df_executions["duration_minutes"] = (
            df_executions["end_time"] - df_executions["start_time"]
        ).dt.total_seconds() / 60.0

        # Format durations to .2f
        df_executions["duration_minutes"] = df_executions["duration_minutes"].apply(
            lambda x: "{:.2f}".format(x)
        )

        # Reorder columns
        cols = [
            "scriptSalIndex",
            "start_time",
            "end_time",
            "duration_minutes",
            "FinalProcessStatus",
            "FinalScriptStatus",
        ]
        df_executions = df_executions[cols]

        # Sort by start time in descending order
        df_executions = df_executions.sort_values("start_time", ascending=False)
    else:
        print("No executions found.")

    return df_executions


## Bump Test Queries and Processing

### Query Bump Logs

In [ ]:
async def query_bump_logs(start_date: str, end_date: str, client_name="summit_efd"):
    """
    Queries the log messages related to bump tests from the EFD.

    Parameters
    ----------
    start_date : str
        Start date of the query in ISO format, e.g. "2024-11-04T12:00:00".
    end_date : str
        End date of the query in ISO format, e.g. "2024-11-04T13:00:00".
    client_name : str, optional
        Name of the EFD client. Defaults to "summit_efd".

    Returns
    -------
    pd.DataFrame
        DataFrame of bump log messages with the requested fields.
    """
    # Convert strings to datetime, then to astropy Time
    start_dt = datetime.fromisoformat(start_date)
    end_dt = datetime.fromisoformat(end_date)

    possible_clients = ["summit_efd", "usdf_efd"]
    if client_name not in possible_clients:
        print(f"Invalid client name. Possible clients: {possible_clients}")
        return None

    client = makeEfdClient(client_name)

    try:
        bump_logs = await client.select_time_series(
            topic_name="lsst.sal.MTM1M3.logevent_logMessage",
            fields=["message"],
            start=Time(start_dt.isoformat(), format="isot", scale="utc"),
            end=Time(end_dt.isoformat(), format="isot", scale="utc"),
        )
        return bump_logs
    except Exception as e:
        print(
            f"Error querying data from {start_dt.isoformat()} to {end_dt.isoformat()}: {e}"
        )
        return pd.DataFrame()


def process_bump_logs(bump_logs, expected_force_range=222, tolerance=5):
    """
    Processes bump log messages to extract relevant information and calculate deviations
    from expected forces.

    Parameters
    ----------
    bump_logs : pd.DataFrame
        DataFrame containing bump log messages.
    expected_force_range : float, optional
        Expected magnitude of the applied force.
    tolerance : float, optional
        Tolerance for the allowed variation in force.

    Returns
    -------
    pd.DataFrame
        Processed DataFrame with extracted and calculated data.
    """
    df_filtered_bump_log = bump_logs[
        bump_logs["message"].str.contains("Failed FA")
    ].copy()

    if df_filtered_bump_log.empty:
        print("No failed FA messages found in bump logs.")
        return pd.DataFrame()

    # Extract relevant info from the message
    df_filtered_bump_log.loc[:, "ID"] = df_filtered_bump_log["message"].str.extract(
        r"FA ID (\d+)"
    )
    orientation_index = df_filtered_bump_log["message"].str.extract(r"\((X|Y|Z)(\d+)\)")
    df_filtered_bump_log.loc[:, "Orientation"] = orientation_index[0]
    df_filtered_bump_log.loc[:, "Index"] = orientation_index[1]
    df_filtered_bump_log.loc[:, "Error Message"] = df_filtered_bump_log[
        "message"
    ].str.extract(r"- (.+)$")

    new_df = df_filtered_bump_log.reset_index().rename(columns={"index": "Time"})[
        ["Time", "ID", "Orientation", "Index", "Error Message"]
    ]

    # Extract measured force (ignore disabled actuators with zero force)
    new_df["MeasuredForce"] = (
        new_df["Error Message"].str.extract(r"\(([\-\d.]+)\)").astype(float)
    )
    new_df = new_df[new_df["MeasuredForce"] != 0]

    # Classify applied force direction
    new_df["AppliedForceDirection"] = new_df["Error Message"].apply(
        lambda x: "Positive" if "measured force plus" in x else "Negative"
    )

    # Calculate deviations
    upper_limit = expected_force_range
    lower_limit = -expected_force_range
    new_df["Deviation"] = 0.0

    for idx, row in new_df.iterrows():
        if row["AppliedForceDirection"] == "Positive":
            new_df.at[idx, "Deviation"] = float(row["MeasuredForce"]) - upper_limit
        elif row["AppliedForceDirection"] == "Negative":
            new_df.at[idx, "Deviation"] = float(row["MeasuredForce"]) - lower_limit

    # Classify deviation as overshoot or undershoot
    def classify_deviation(row):
        deviation = abs(row["MeasuredForce"]) - expected_force_range
        if deviation > 0:
            return "Overshoot"
        else:
            return "Undershoot"

    new_df["DeviationType"] = new_df.apply(classify_deviation, axis=1)

    cols = [
        "Time",
        "ID",
        "Orientation",
        "Index",
        "AppliedForceDirection",
        "MeasuredForce",
        "Deviation",
        "DeviationType",
    ]
    return new_df[cols]

### Plotting and Visualization of Bump Logs

In [ ]:
m1m3_actuator_id_index_table = {fa.actuator_id: fa.index for fa in FATable}


def get_m1m3_actuator_ids():
    """Get a list of the M1M3 actuator ids."""
    return list(m1m3_actuator_id_index_table.keys())


def get_xy_position(actuator_list=FATable):
    xpos = [actuator.x_position for actuator in actuator_list]
    ypos = [actuator.y_position for actuator in actuator_list]
    return xpos, ypos


def ActuatorsLayout(ax, df, actuator_list=FATable):
    """
    Plot the layout of M1M3 actuators and highlight failed actuators.
    """
    orientation_colors = {"X": "blue", "Y": "red", "Z": "green"}
    combined_orientations_colors = {"XZ": "orange", "YZ": "black"}

    ax.set_xlabel("X position (m)")
    ax.set_ylabel("Y position (m)")
    ax.set_title("Failures Distribution", fontsize=12)

    ids = get_m1m3_actuator_ids()
    xpos, ypos = get_xy_position(actuator_list)

    actuator_orientations = defaultdict(set)
    for actuator_id, orientation in zip(df["ID"], df["Orientation"]):
        actuator_orientations[int(actuator_id)].add(orientation)

    ax.plot(xpos, ypos, "o", ms=14, color="blue", alpha=0.05, mec="red")
    for l, x, y in zip(ids, xpos, ypos):
        ax.annotate(
            l,
            (x, y),
            textcoords="offset points",
            xytext=(-5.5, -2),
            color="blue",
            size="xx-small",
        )

    for actuator_id, orientations in actuator_orientations.items():
        if actuator_id in m1m3_actuator_id_index_table:
            index = m1m3_actuator_id_index_table[actuator_id]
            if len(orientations) > 1:
                color = combined_orientations_colors["".join(sorted(orientations))]
            else:
                color = orientation_colors[list(orientations)[0]]
            ax.scatter(
                xpos[index],
                ypos[index],
                marker="o",
                facecolors="none",
                edgecolors=color,
                s=250,
                alpha=0.5,
                linewidths=2,
            )

    # Legend
    unique_orientations = {
        ori
        for orientations in actuator_orientations.values()
        if len(orientations) == 1
        for ori in orientations
    }
    combined_orientations = {
        "".join(sorted(orientations))
        for orientations in actuator_orientations.values()
        if len(orientations) > 1
    }

    for orientation in unique_orientations:
        ax.scatter(
            [],
            [],
            marker="o",
            linestyle="None",
            s=10,
            facecolor="none",
            edgecolor=orientation_colors[orientation],
            alpha=0.9,
            label=f"Orientation {orientation}",
        )

    for combined_orientation in combined_orientations:
        ax.scatter(
            [],
            [],
            marker="o",
            linestyle="None",
            s=10,
            facecolor="none",
            edgecolor=combined_orientations_colors[combined_orientation],
            alpha=0.5,
            label=f"Orientation {combined_orientation}",
        )
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))


def plot_deviations_and_layout(
    df_failures, actuator_list=FATable, expected_force=222, tolerance=5
):
    """
    Creates a single figure with:
      - Two subplots stacked vertically (Positive, Negative) on the left,
      - One subplot on the right for the actuator layout, with a square aspect.

    Assumes df_failures already has columns:
      'ID', 'DeviationType', 'AppliedForceDirection', 'MeasuredForce',
      and any others needed (e.g. 'Orientation').
    """

    if df_failures.empty:
        print("The DataFrame is empty. Nothing to plot.")
        return

    # Convert ID to string for plotting on x-axis
    df_failures["ID"] = df_failures["ID"].astype(str)
    sorted_ids = df_failures["ID"].sort_values().unique()

    # Dictionary for how each deviation type should look
    deviation_styles = {
        "Overshoot": {"color": "red", "marker": "o"},
        "Undershoot": {"color": "green", "marker": "s"},
    }

    # Map each ID to a position on the x-axis
    id_to_position = {id_: pos for pos, id_ in enumerate(sorted_ids)}
    df_failures["ID_Pos"] = df_failures["ID"].map(id_to_position)

    # Create the figure and GridSpec layout
    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(
        nrows=2,
        ncols=2,
        width_ratios=[2, 1],  # left is twice as wide as right
        height_ratios=[2, 2],
    )

    # Left column subplots
    ax_positive = fig.add_subplot(gs[0, 0])
    ax_negative = fig.add_subplot(gs[1, 0], sharex=ax_positive)
    # Right column subplot (spans both rows)
    ax_layout = fig.add_subplot(gs[:, 1])

    # Separate the DataFrame
    df_positive = df_failures[df_failures["AppliedForceDirection"] == "Positive"]
    df_negative = df_failures[df_failures["AppliedForceDirection"] == "Negative"]

    # --- Subplot: Positive Forces ---
    for dev_type, group in df_positive.groupby("DeviationType"):
        style = deviation_styles.get(dev_type, {"color": "gray", "marker": "o"})
        ax_positive.scatter(
            group["ID_Pos"],
            group["MeasuredForce"],
            c=style["color"],
            marker=style["marker"],
            alpha=0.7,
            label=f"Positive - {dev_type}",
        )
    # Expected lines & tolerance range
    ax_positive.axhline(expected_force, color="gray", linestyle="--", linewidth=0.8)
    ax_positive.fill_between(
        [-1, len(sorted_ids)],
        expected_force - tolerance,
        expected_force + tolerance,
        color="gray",
        alpha=0.2,
        label="Tolerance Range",
    )
    ax_positive.set_ylabel("Measured Force (N)")
    ax_positive.set_title("Measured Forces (Positive)")

    # --- Subplot: Negative Forces ---
    for dev_type, group in df_negative.groupby("DeviationType"):
        style = deviation_styles.get(dev_type, {"color": "gray", "marker": "o"})
        ax_negative.scatter(
            group["ID_Pos"],
            group["MeasuredForce"],
            c=style["color"],
            marker=style["marker"],
            alpha=0.7,
            label=f"Negative - {dev_type}",
        )
    ax_negative.axhline(-expected_force, color="gray", linestyle="--", linewidth=0.8)
    ax_negative.fill_between(
        [-1, len(sorted_ids)],
        -expected_force - tolerance,
        -expected_force + tolerance,
        color="gray",
        alpha=0.2,
    )
    ax_negative.set_xlabel("FA ID")
    ax_negative.set_ylabel("Measured Force (N)")
    ax_negative.set_title("Measured Forces (Negative)")

    # Set x-ticks only on the bottom subplot
    ax_negative.set_xticks(range(len(sorted_ids)))
    ax_negative.set_xticklabels(sorted_ids, rotation=45, ha="center")

    # Custom legend in the top subplot
    legend_handles = []
    for dev_type, style in deviation_styles.items():
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker=style["marker"],
                color="w",
                label=dev_type,
                markerfacecolor=style["color"],
                markersize=8,
            )
        )
    ax_positive.legend(handles=legend_handles, title="Deviation Type", loc="upper left")

    # --- Right subplot: Layout of Actuators ---
    ActuatorsLayout(ax_layout, df_failures, actuator_list=actuator_list)

    # Ensure the layout subplot is square
    ax_layout.set_aspect("equal", adjustable="box")

    # Final adjustments
    plt.tight_layout()


def get_failed_actuator_ids(processed_bump_logs):
    try:
        failed_actuator_ids = processed_bump_logs["ID"].astype(int).unique()
        if len(failed_actuator_ids) == 0:
            raise ValueError("No failed actuators found in the processed bump logs.")
        return failed_actuator_ids
    except KeyError as e:
        print(f"KeyError: {e}. Ensure the DataFrame contains the 'ID' column.")
    except Exception as e:
        print(f"An error occurred: {e}")

## Detailed Force Actuator Bump Analysis

### Internal Helper Functions for Bump Analysis

In [ ]:
def max_error(errors):
    return np.max([np.max(errors), np.max(errors * -1.0)])


def rms_times(t_start_str: str):
    """
    Return times where RMS calculation is performed, depending on a date threshold.
    """
    # This is the date of the IRQ change, which changed the bump test timings
    change_date = Time("2024-10-12T00:00:00", format="isot", scale="utc")
    t_start = Time(t_start_str, format="isot", scale="utc")

    if t_start < change_date:
        rms_t1 = 3.0
        rms_t2 = 4.0
        rms_t3 = 10.0
        rms_t4 = 11.0
    else:
        rms_t1 = 2.9
        rms_t2 = 3.9
        rms_t3 = 9.3
        rms_t4 = 10.3
    return [rms_t1, rms_t2, rms_t3, rms_t4]


def rms_error(times, errors, _rms_times):
    [rms_t1, rms_t2, rms_t3, rms_t4] = _rms_times
    error = 0.0
    num = 0
    for i, t in enumerate(times):
        if (t > rms_t1 and t < rms_t2) or (t > rms_t3 and t < rms_t4):
            num += 1
            error += errors[i] ** 2
    if num == 0:
        return np.nan
    else:
        return np.sqrt(error / num)


async def plot_bumps_and_errors(
    axs,
    client,
    bump,
    bt_result,
    force,
    follow,
    applied,
    p_s,
    actuator_id,
    start_time_str,
    end_time_str,
):
    BUMP_TEST_DURATION = 14.0
    max_x_ticks = 25

    measured_forces_times = []
    measured_forces_values = []
    following_error_values = []
    applied_forces_times = []
    applied_forces_values = []
    t_starts = []

    # Determine which column in axs to use
    if p_s == "Primary":
        plot_index = 0
    else:
        plot_index = 1

    # Filter DataFrame for tests that are in TESTINGPOSITIVE state
    results = bt_result[bt_result[bump] == BumpTest.TESTINGPOSITIVE]
    for bt_index in range(len(results)):
        # Convert TAI (unix_tai) to astropy Time
        t_start = Time(
            results["timestamp"].values[bt_index] - 1.0,
            format="unix_tai",
            scale="tai",
        )
        t_starts.append(t_start.isot)
        t_end = t_start + TimeDelta(BUMP_TEST_DURATION, format="sec")

        measured_forces = await client.select_time_series(
            "lsst.sal.MTM1M3.forceActuatorData",
            [force, follow, "timestamp"],
            t_start.utc,
            t_end.utc,
        )
        applied_forces = await client.select_time_series(
            "lsst.sal.MTM1M3.appliedForces",
            [applied, "timestamp"],
            t_start.utc,
            t_end.utc,
        )

        # Shift time zero
        t0 = measured_forces["timestamp"].values[0]
        measured_forces["timestamp"] -= t0
        applied_forces["timestamp"] -= t0

        measured_forces_times.append(measured_forces["timestamp"].values)
        measured_forces_values.append(measured_forces[force].values)
        following_error_values.append(measured_forces[follow].values)
        applied_forces_times.append(applied_forces["timestamp"].values)
        applied_forces_values.append(applied_forces[applied].values)

    # ─────────────────────────────
    #  First Row: Forces vs Time
    # ─────────────────────────────
    axs[0][plot_index].set_title(f"Actuator {actuator_id} {p_s} forces vs time")

    # Plot the first test’s applied force in some default style
    axs[0][plot_index].plot(
        applied_forces_times[0], applied_forces_values[0], color="gray"
    )

    # For all but the last test, use some default style
    for i in range(len(measured_forces_times) - 1):
        axs[0][plot_index].plot(
            measured_forces_times[i],
            measured_forces_values[i],
            color="gray",
            linewidth=1.0,
        )

    # Plot the most recent test (last in the list) in black & thicker line
    i_last = len(measured_forces_times) - 1
    if i_last >= 0:
        axs[0][plot_index].plot(
            measured_forces_times[i_last],
            measured_forces_values[i_last],
            color="black",
            linewidth=2.0,
        )
        # If you still want to see the most recent *applied* force in black:
        axs[0][plot_index].plot(
            applied_forces_times[i_last],
            applied_forces_values[i_last],
            color="black",
            linewidth=1.5,
            linestyle="--",
        )

    # No legend in the first row
    axs[0][plot_index].set_xlim(0, BUMP_TEST_DURATION)
    axs[0][plot_index].set_xlabel("Time (sec.)")
    axs[0][plot_index].set_ylabel("Force (N)")

    # ─────────────────────────────
    #  Second Row: Last Test Only
    # ─────────────────────────────
    axs[1][plot_index].set_title(
        f"Actuator {actuator_id} {p_s} forces {t_starts[-1] if t_starts else ''}"
    )

    if i_last >= 0:
        axs[1][plot_index].plot(
            applied_forces_times[i_last],
            applied_forces_values[i_last],
            label="Commanded force",
        )
        axs[1][plot_index].plot(
            measured_forces_times[i_last],
            measured_forces_values[i_last],
            label="Measured force",
        )
        axs[1][plot_index].plot(
            measured_forces_times[i_last],
            following_error_values[i_last],
            label="Following error",
        )

    axs[1][plot_index].set_xlim(0, BUMP_TEST_DURATION)
    axs[1][plot_index].set_xlabel("Time (sec.)")
    axs[1][plot_index].set_ylabel("Force (N)")

    # Show vertical lines for RMS windows
    if t_starts:
        _rms_t = rms_times(t_starts[i_last])
        for marker in _rms_t:
            axs[1][plot_index].axvline(marker, ls="--", color="black")

    # Grab existing handles/labels once, remove duplicates, and show the last one
    lines, labels = axs[1][plot_index].get_legend_handles_labels()
    unique_labels = {}
    for line, label in zip(lines, labels):
        unique_labels[label] = line  # This will overwrite and keep the last occurrence
    axs[1][plot_index].legend(unique_labels.values(), unique_labels.keys(), loc="best")

    # ─────────────────────────────
    #  Third Row: Following Errors
    # ─────────────────────────────
    axs[2][plot_index].set_title(f"Actuator {actuator_id} {p_s} following errors")
    times = []
    max_errors = []
    rms_errors = []

    for i in range(len(measured_forces_times)):
        times.append(t_starts[i])
        max_errors.append(max_error(following_error_values[i]))
        times_rms = rms_times(t_starts[i])
        rms_val = rms_error(
            measured_forces_times[i], following_error_values[i], times_rms
        )
        rms_errors.append(rms_val)

    axs[2][plot_index].plot(times, rms_errors, marker="x", color="blue", label="RMS")
    axs[2][plot_index].plot(times, max_errors, marker="+", color="green", label="Max")

    axs[2][plot_index].set_ylim(0, 1000)
    axs[2][plot_index].set_yscale("symlog", linthresh=10)
    axs[2][plot_index].set_yticks([0, 2, 4, 6, 8, 10, 50, 100, 500, 1000])
    axs[2][plot_index].yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    axs[2][plot_index].tick_params(axis="x", rotation=90)
    axs[2][plot_index].set_ylabel("RMS and Max errors (N)")

    # Remove duplicate legend labels
    lines3, labels3 = axs[2][plot_index].get_legend_handles_labels()
    unique_labels3 = {}
    for line, label in zip(lines3, labels3):
        if label not in unique_labels3:
            unique_labels3[label] = line
    axs[2][plot_index].legend(
        unique_labels3.values(), unique_labels3.keys(), loc="best"
    )


async def plot_actuator_error(
    fig, client, fa_id, bt_results, start_time_str, end_time_str
):
    axs = fig.subplots(3, 2)
    plt.gcf().subplots_adjust(bottom=0.25, wspace=0.3, hspace=0.3)

    fa_data = force_actuator_from_id(fa_id)
    bt_result = bt_results[bt_results["actuatorId"] == fa_id]

    # Query the past 5 days data
    # (Again, end_time_str is guaranteed isoformat now)
    past_days_data = await query_past_days_data(client, fa_id, end_time_str, days=ndays)

    # Plot the "past 5 days" data
    await plot_bumps_and_errors(
        axs,
        client,
        f"primaryTest{fa_data.index}",
        past_days_data,
        f"primaryCylinderForce{fa_data.index}",
        f"primaryCylinderFollowingError{fa_data.index}",
        f"zForces{fa_data.z_index}",
        "Primary",
        fa_id,
        start_time_str,
        end_time_str,
    )

    if fa_data.actuator_type.name == "DAA":
        await plot_bumps_and_errors(
            axs,
            client,
            f"secondaryTest{fa_data.s_index}",
            past_days_data,
            f"secondaryCylinderForce{fa_data.s_index}",
            f"secondaryCylinderFollowingError{fa_data.s_index}",
            f"zForces{fa_data.z_index}",
            "Secondary",
            fa_id,
            start_time_str,
            end_time_str,
        )
    else:
        axs.flat[1].set_title("Not a DAA actuator")
        axs.flat[3].set_title("Not a DAA actuator")
        axs.flat[5].set_title("Not a DAA actuator")

    # Then plot actual bumps from the selected range
    bump = f"primaryTest{fa_data.index}"
    force = f"primaryCylinderForce{fa_data.index}"
    follow = f"primaryCylinderFollowingError{fa_data.index}"
    applied = f"zForces{fa_data.z_index}"

    await plot_bumps_and_errors(
        axs,
        client,
        bump,
        bt_result,
        force,
        follow,
        applied,
        "Primary",
        fa_id,
        start_time_str,
        end_time_str,
    )

    if fa_data.actuator_type.name == "DAA":
        bump = f"secondaryTest{fa_data.s_index}"
        force = f"secondaryCylinderForce{fa_data.s_index}"
        follow = f"secondaryCylinderFollowingError{fa_data.s_index}"
        secondary_name = fa_data.orientation.name
        if secondary_name in ["X_PLUS", "X_MINUS"]:
            applied = f"xForces{fa_data.x_index}"
        elif secondary_name in ["Y_PLUS", "Y_MINUS"]:
            applied = f"yForces{fa_data.y_index}"
        else:
            applied = f"zForces{fa_data.z_index}"

        await plot_bumps_and_errors(
            axs,
            client,
            bump,
            bt_result,
            force,
            follow,
            applied,
            secondary_name,
            fa_id,
            start_time_str,
            end_time_str,
        )

    plt.show()


async def query_past_days_data(client, fa_id, end_time_str: str, days=5):
    """
    Query bump test status data for a specific actuator ID, from end_time_str going back 'days'.

    Parameters
    ----------
    client : EfdClient
        The EFD client to query data.
    fa_id : int
        The force actuator ID.
    end_time_str : str
        End time in ISO format, e.g. "2024-12-07T17:10:17.601".
    days : float
        How many days in the past to retrieve.

    Returns
    -------
    pd.DataFrame
        DataFrame with the bump test status results for the given time window.
    """
    end_time = Time(end_time_str, format="isot", scale="utc")
    start_time = end_time - TimeDelta(days, format="jd")

    bump_n_days_results = await client.select_time_series(
        "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatus",
        "*",
        start_time,
        end_time,
    )
    return bump_n_days_results

## Getting Past Executions

### Script Queue Logs
Here we create a summary of the last few executions of the `check_actuators.py` script. 

In [ ]:
past_hours = convert_to_hours(past_time)

now_utc = datetime.utcnow()
end_dt = now_utc
start_dt = now_utc - timedelta(hours=past_hours)

start_str = start_dt.isoformat()
end_str = end_dt.isoformat()

script_logs = await query_script_queue_logs(
    start_str, end_str, client_name=notebook_client_name
)
script_logs_processed = filter_and_process_queue_logs(script_logs)
df_executions = extract_execution_details(script_logs_processed)

### Interactive Widget to Query and Plot Bump Test Failures

In the section, a set of **Jupyter Widgets** is provided to guide you through the bump test log analysis:

1. **`SalIndex` Dropdown:**
   Lets you select a specific script execution (identified by its `salIndex`) carried in the time range provided. Once selected, the notebook retrieves the associated execution details (start time, end time, duration, etc.). Note that executions shown here limited to the provided time range.

2. **`EFD Client` Dropdown:**
   Allows switching between available EFD clients (such as `usdf_efd` or `summit_efd`) if you need to query data from different environments. If doing analysis on real-time, the `summit_efd` might be the preferred client.

3. **“Get FA results” Button:**
   Initiates the actual query for bump test logs within the selected script execution’s time window. The results are then processed to identify any failed force actuators.

4. **`FA ID` Dropdown:**
   Automatically populated with the list of failed actuator IDs found in the processed logs. Selecting an actuator ID triggers the retrieval of its forces, following errors, and other relevant data from the EFD. The notebook will then display diagnostic plots that focus on that specific actuator.

This widget-based workflow makes it easy to iterate through different script executions, check the resulting failed actuators, and visualize each one’s performance.


In [ ]:
# Display executions summary
if df_executions.empty:
    print("No executions found in the last {:.1f} days.".format(past_hours / 24.0))
else:
    print("Executions found in the last {:.1f} days.".format(past_hours / 24.0))
    display(df_executions)

In [ ]:
%matplotlib inline

# -------------------------------------------------------------------
# Assumptions:
#   The following variables and functions are defined *elsewhere*
#   in the notebook or imported from external modules:
#   - df_executions: DataFrame listing script executions
#   - notebook_client_name: default EFD client name
#   - makeEfdClient, query_bump_logs, process_bump_logs:
#       helper functions to query and parse logs from the EFD
#   - get_failed_actuator_ids: extracts ID values from processed logs
#   - plot_actuator_error: plots detailed error data for a single FA ID
#   - plot_deviations_and_layout: a combined plot for measured forces
#     (positive/negative) and the layout distribution
# -------------------------------------------------------------------

################################################################################
# A global variable to remember the old callback function, so we can remove it
# when a new execution is selected. This ensures we don't double-trigger events.
################################################################################
previous_fa_id_callback = None

################################################################################
# 1) Define the interactive widgets
################################################################################

# Dropdown for picking a scriptSalIndex (execution) from df_executions
script_sal_index_selector = widgets.Dropdown(
    options=df_executions["scriptSalIndex"].unique(),
    value=df_executions["scriptSalIndex"].iloc[-1],  # default to the last one
    description="SalIndex:",
    disabled=False,
)

# Dropdown for choosing which EFD client to use (summit_efd, usdf_efd, etc.)
client_selector = widgets.Dropdown(
    options=["usdf_efd", "summit_efd"],
    value=notebook_client_name,
    description="EFD Client:",
    disabled=False,
)

# Checkbox to decide whether to plot combined measured forces and layout
show_forces_checkbox = widgets.Checkbox(
    value=False, description="Plot measured forces", disabled=False
)

# Button that triggers the main query workflow
query_button = widgets.Button(
    description="Get FA Results",
    disabled=False,
    button_style="info",
    tooltip="Click to query/process bump logs",
    icon="search",
)

# Dropdown for selecting a Force Actuator (FA) ID to plot detailed errors
fa_id_selector = widgets.Dropdown(
    options=[],
    description="FA ID:",
    disabled=False,
)

# Combine all widgets into a vertical layout, with the query button and
# checkbox side by side (HBox)
ui = widgets.VBox(
    [
        client_selector,
        script_sal_index_selector,
        widgets.HBox([query_button, show_forces_checkbox]),
        fa_id_selector,
    ]
)

################################################################################
# 2) Define the async callback for the "Get FA Results" button
################################################################################


async def on_query_button_clicked(b):
    """
    Handles the entire workflow when the user clicks the "Get FA Results" button.
    1. Clears previous output and redisplays the UI.
    2. Reads the selected execution from 'script_sal_index_selector'.
    3. Queries and processes bump logs from the EFD.
    4. Optionally plots measured forces if checkbox is checked.
    5. Displays summary of processed logs.
    6. Populates 'fa_id_selector' with new failed actuator IDs.
    7. Sets up a new callback for detailed FA ID plots.
    8. Automatically triggers the first ID in the list to plot immediately.
    """

    global previous_fa_id_callback

    # -----------------------------------------------------------------------
    # Clear the entire cell output each time the user clicks "Get FA Results"
    # This ensures a fresh start for logs and plots of the *new* execution.
    # -----------------------------------------------------------------------
    clear_output(wait=True)
    display(ui)

    # Gather user selections: scriptSalIndex + EFD client
    selected_script_sal_index = script_sal_index_selector.value
    execution_row = df_executions[
        df_executions["scriptSalIndex"] == selected_script_sal_index
    ].iloc[0]

    # Extract start/end times, durations, final statuses, etc.
    t_start = execution_row["start_time"]
    t_end = execution_row["end_time"]
    client_name = client_selector.value
    duration = execution_row["duration_minutes"]
    process_status = execution_row["FinalProcessStatus"]
    script_status = execution_row["FinalScriptStatus"]

    # Print basic info about the chosen execution
    print(
        f"Execution details:\n"
        f"    Script SalIndex: {selected_script_sal_index}\n"
        f"    Start Time: {t_start}\n"
        f"    End Time:   {t_end}\n"
        f"    Duration:   {duration} minutes\n"
        f"    Process Final Status: {process_status}\n"
        f"    Script Final Status:  {script_status}\n"
    )

    # Convert times to ISO format for astropy queries
    start_time_str = t_start.isoformat()
    end_time_str = t_end.isoformat()

    # If we had a previous callback from an older execution, unobserve it
    # to prevent multiple triggers.
    if previous_fa_id_callback is not None:
        fa_id_selector.unobserve(previous_fa_id_callback, names="value")
        previous_fa_id_callback = None

    # Create the EFD client to query logs
    client = makeEfdClient(client_name)

    print("Querying and processing bump log errors...")
    # Query the raw bump logs from the EFD
    bump_logs = await query_bump_logs(start_time_str, end_time_str, client_name)
    if bump_logs.empty:
        print("No bump test error logs found for the selected execution.")
        return

    # Process the logs (extract ID, DeviationType, etc.)
    processed_bump_logs = process_bump_logs(bump_logs)
    if processed_bump_logs.empty:
        print("No failed actuators found in the bump test logs.")
        return

    # If the user wants to see measured forces & layout, plot them now
    if show_forces_checkbox.value:
        print("\nMeasured positive/negative forces and failures distribution:")
        plot_deviations_and_layout(processed_bump_logs)
        plt.show()

    # Show the summary DataFrame of processed bump logs
    print("\nBump Test Error Summary:")
    display(processed_bump_logs)

    # Extract the IDs of failed actuators and populate the FA ID dropdown
    try:
        failed_actuator_ids = get_failed_actuator_ids(processed_bump_logs)
    except Exception as e:
        print(f"Error: {e}")
        return
    fa_id_selector.options = failed_actuator_ids

    # Query the bump test status data for more details
    bump_results = await makeEfdClient(client_name).select_time_series(
        "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatus",
        "*",
        Time(start_time_str, format="isot", scale="utc"),
        Time(end_time_str, format="isot", scale="utc"),
    )
    if bump_results.empty:
        print("No bump test status data found for the selected execution.")
        return

    # Define a nested async function that plots error data for a specific FA ID
    async def plot_selected_actuator_errors(change):
        """
        Called when the user picks a new FA ID in 'fa_id_selector'.
        Plots detailed bump test error data for the chosen actuator.
        """
        fa_id = change["new"]
        # If 'fa_id' is None, ignore.
        if fa_id is None:
            return

        print(
            f"\nPlotting errors for Actuator ID: {fa_id}. "
            f"This may take a while, especially if 'ndays' > 5."
        )
        fig = plt.figure(figsize=(15, 20))
        await plot_actuator_error(
            fig, client, fa_id, bump_results, start_time_str, end_time_str
        )

    # This function is attached to the dropdown changes (value changes).
    def on_fa_id_change(change):
        asyncio.ensure_future(plot_selected_actuator_errors(change))

    # Observe the new callback
    fa_id_selector.observe(on_fa_id_change, names="value")
    # Store it so we can unobserve it next time
    previous_fa_id_callback = on_fa_id_change

    # Force a "real" change event to auto-plot the first ID.
    # 1) set to None so the widget sees a genuine change next
    # 2) set to the first item in the new list
    if len(fa_id_selector.options) > 0:
        fa_id_selector.value = None
        fa_id_selector.value = fa_id_selector.options[0]


# Attach the main callback to the "Get FA Results" button
query_button.on_click(lambda b: asyncio.ensure_future(on_query_button_clicked(b)))

# Finally, display the UI so the user can interact
display(ui)

## Debugging / Testing Cells

### Manual Bump Log Debug

In [ ]:
_start_date = "2024-12-07T16:04:07.939"
_end_date = "2024-12-07T17:10:17.601"

client = makeEfdClient(notebook_client_name)

print(
    f"Querying bump logs from {_start_date} to {_end_date} for client {notebook_client_name}"
)
try:
    _bump_logs = await client.select_time_series(
        topic_name="lsst.sal.MTM1M3.logevent_logMessage",
        fields=["message"],
        start=Time(_start_date, format="isot", scale="utc"),
        end=Time(_end_date, format="isot", scale="utc"),
    )
except Exception as e:
    print(f"Error querying data from {_start_date} to {_end_date}: {e}")

_processed_bump_logs = process_bump_logs(_bump_logs)

_bump_results = await client.select_time_series(
    "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatus",
    "*",
    Time(_start_date, format="isot", scale="utc"),
    Time(_end_date, format="isot", scale="utc"),
)


# fig = plt.figure(figsize=(15, 20))
# await plot_actuator_error(fig, client, 106, _bump_results, _start_date, _end_date)
# plt.show()